# 🎫 Customer Support Ticket Tagger — Explained

**What this notebook does:** fine-tunes a BERT-family model (**DeBERTa-v3**) on a Kaggle dataset of support tickets so it can automatically **route a ticket to the right department** (one of 10 queues).

**The flow:**
`load data → turn labels into numbers → split train/test → load model → tokenize → train (10 epochs) → evaluate → save → test predictions`

Each code cell below has a plain-English explanation right above it.

---

### 📦 Cell 0 — Install the libraries

Installs the 4 Python packages this notebook needs (run once):
- `transformers` → load & train the AI model (Hugging Face)
- `datasets` → load and prepare the data
- `torch` → the math engine (PyTorch) that runs the model
- `scikit-learn` → helper tools (turn labels into numbers, split data, score accuracy)

*(On Kaggle/Colab these are often pre-installed, so the line may be commented out with `#`.)*

In [ ]:
pip install transformers datasets torch scikit-learn

## Importing necessary libraries

### 🧰 Cell 2 — Import the tools

- `import numpy as np` → math/array library (`np` is its short nickname)
- `import pandas as pd` → tables/spreadsheets in Python (`pd`) — used to hold the CSV
- `from sklearn.preprocessing import LabelEncoder` → tool that turns text labels (e.g. "Billing") into numbers
- `from datasets import Dataset` → Hugging Face's data container (the format the model trainer expects)

In [2]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from datasets import Dataset

### 📄 Cell 3 — Load the dataset

- `df = pd.read_csv(".../customer_tickets.csv")` → read the CSV file into a table called `df`
- `df.columns = ["text","labels"]` → rename the 2 columns to `text` (the ticket) and `labels` (the department)
- `df.head()` → show the first 5 rows so you can eyeball the data

In [3]:
df = pd.read_csv("/kaggle/input/customer-support-ticket-tagging/customer_tickets.csv")
df.columns = ["text","labels"]
df.head()

,text,labels
0,"Dear Customer Support Team, We are experiencin...",Technical Support
1,"Dear Customer Support,<br><br>I hope this mess...",Product Support
2,"Dear Tech Online Store Customer Support,\n\nI ...",Returns and Exchanges
3,"Dear IT Services Customer Support, \n\nWe are ...",Product Support
4,"Greetings IT Services Customer Support,\n\nI a...",Technical Support


### 🧹 Cell 4 — Clean the data

- `df.dropna(inplace=True)` → **drop** any rows that have **na** (missing/empty) values.
  - `inplace=True` = change `df` directly (don't make a copy).
  - Why: empty rows would break training.

In [4]:
df.dropna(inplace=True)

### 🔢 Cell 5 — Turn labels into numbers + split the data

The model can't read words like "Billing" — it needs numbers. And we need separate data to train on vs. test on.
- `label_encoder = LabelEncoder()` → create the text→number converter
- `df['labels'] = label_encoder.fit_transform(df['labels'])` → convert every department name to a number (e.g. "Billing and Payments" → 0)
- `dataset = Dataset.from_pandas(df[['text','labels']])` → convert the pandas table into a Hugging Face `Dataset`
- `hf_dataset = dataset.train_test_split(test_size=0.145)` → split: ~85% for **training**, ~14.5% for **testing** (288 train / 50 test)
- `print(hf_dataset)` → show the split result

In [5]:
label_encoder = LabelEncoder()
df['labels'] = label_encoder.fit_transform(df['labels']) # converts labels which are in character to numerix format

# Convert to Hugging Face Dataset
dataset = Dataset.from_pandas(df[['text', 'labels']])
hf_dataset = dataset.train_test_split(test_size=0.145)
print(hf_dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', '__index_level_0__'],
        num_rows: 288
    })
    test: Dataset({
        features: ['text', 'labels', '__index_level_0__'],
        num_rows: 50
    })
})


### 🏷️ Cell 6 — See the label list

- `label_encoder.classes_` → shows the list of the 10 department names, **in order**.
  - The position in this list = the number each got (index 0 = "Billing and Payments", index 9 = "Technical Support").
  - This is how we'll later turn a predicted **number** back into a **department name**.

In [6]:
label_encoder.classes_

array(['Billing and Payments', 'Customer Service', 'General Inquiry',
       'Human Resources', 'IT Support', 'Product Support',
       'Returns and Exchanges', 'Sales and Pre-Sales',
       'Service Outages and Maintenance', 'Technical Support'],
      dtype=object)

### 🤖 Cell 7 — Load the model + tokenizer

- `from transformers import AutoTokenizer, AutoModelForSequenceClassification` → import the loaders
- `model_name = "microsoft/deberta-v3-base"` → pick **DeBERTa-v3** (a BERT-family **encoder** model, great for classification)
- `tokenizer = AutoTokenizer.from_pretrained(model_name)` → load its matching tokenizer (text → tokens)
- `model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=len(label_encoder.classes_))`
  - Loads the model with a **classification head** sized to **10 labels** (our 10 departments).
  - ⚠️ The warning "*weights ... newly initialized ... you should TRAIN this model*" is **normal** — the classification head starts blank and gets trained in the next cells.

In [7]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "microsoft/deberta-v3-base"  # loading the deberta model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=len(label_encoder.classes_))

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/convert_slow_tokenizer.py:551: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


### ✂️ Cell 8 — Tokenize the text

- `def preprocess_function(examples):` → a small function that tokenizes a batch of tickets
- `return tokenizer(examples['text'], truncation=True, padding=True)`
  - `truncation=True` → cut off text that's too long
  - `padding=True` → add filler so all tickets in a batch are the same length
- `tokenized_datasets = hf_dataset.map(preprocess_function, batched=True)` → apply that function to **all** rows (train + test), in batches

In [8]:
def preprocess_function(examples):
    return tokenizer(examples['text'], truncation=True, padding=True)

tokenized_datasets = hf_dataset.map(preprocess_function, batched=True)

Map:   0%|          | 0/288 [00:00<?, ? examples/s]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Map:   0%|          | 0/50 [00:00<?, ? examples/s]

### ⚙️ Cell 9 — Set the training settings + build the Trainer

`TrainingArguments` = all the knobs for how training runs:
- `output_dir="./results"` → where to save results
- `eval_strategy="steps"` → check test performance periodically during training
- `learning_rate=0.00002` → how big each learning step is (small = careful learning)
- `per_device_train_batch_size=8` / `eval_batch_size=8` → process 8 tickets at a time
- `num_train_epochs=10` → go through the whole training set **10 times**
- `weight_decay=0.05` → a setting that helps prevent overfitting (memorizing)
- `logging_dir` / `logging_steps=30` → where/how often to log progress
- `report_to='none'` → don't send logs to external tools

`Trainer(...)` = bundles everything (model + settings + train data + test data + tokenizer) into one object that runs the training.

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="steps",
    learning_rate=0.00002,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=10,
    weight_decay=0.05,
    logging_dir='./logs',
    logging_steps=30,
    report_to='none'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['test'],
    processing_class=tokenizer
)

### 🏋️ Cell 10 — Train the model

- `trainer.train()` → **the actual learning happens here.** The model reads the training tickets 10 times and adjusts its weights to predict the right department.
- The output shows `training_loss` — **lower = better**. (Here it ends around 1.60, which is still fairly high → the model learned only partially, mainly because the dataset is small.)

In [10]:
trainer.train()

/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '


Step,Training Loss,Validation Loss
30,1.977500,1.886748
60,1.790000,1.799954
90,1.705100,1.639473
120,1.517500,1.475695
150,1.357500,1.409584
180,1.250800,1.389830


TrainOutput(global_step=180, training_loss=1.5997306823730468, metrics={'train_runtime': 176.3015, 'train_samples_per_second': 16.336, 'train_steps_per_second': 1.021, 'total_flos': 538768242754560.0, 'train_loss': 1.5997306823730468, 'epoch': 10.0})

### 📊 Cell 11 — Evaluate on the test set

- `trainer.evaluate()` → test the trained model on the **50 unseen** test tickets.
- Shows `eval_loss` (~1.39 here) — again, lower is better. It tells you how well the model generalizes to data it didn't train on.

In [11]:
trainer.evaluate()

/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '


{'eval_loss': 1.3898301124572754,
 'eval_runtime': 1.1449,
 'eval_samples_per_second': 43.672,
 'eval_steps_per_second': 3.494,
 'epoch': 10.0}

### 💾 Cell 12 — Save the trained model

- `trainer.save_model("./text-classification-model")` → save the fine-tuned **model** to a folder
- `tokenizer.save_pretrained("./text-classification-model")` → save the **tokenizer** alongside it
  - Why both: you always need the matching tokenizer to use a model later.

In [12]:
trainer.save_model("./text-classification-model")
tokenizer.save_pretrained("./text-classification-model")

('./text-classification-model/tokenizer_config.json',
 './text-classification-model/special_tokens_map.json',
 './text-classification-model/spm.model',
 './text-classification-model/added_tokens.json',
 './text-classification-model/tokenizer.json')

### 🚀 Cell 13 — Load the saved model as a pipeline

- `from transformers import pipeline` → import the easy all-in-one helper
- `classifier = pipeline("text-classification", model="./text-classification-model", tokenizer=tokenizer)`
  - Loads **your saved model** back and wraps it into a ready-to-use `classifier`.
  - Now you can just call `classifier("some ticket text")` to get a predicted department.
  - *(The "Model will be on CPU" message just means it's not using the GPU — fine for a few predictions.)*

In [13]:
from transformers import pipeline
# loading the locally saved model
classifier = pipeline("text-classification", model="./text-classification-model", tokenizer=tokenizer)

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


### 🧪 Cell 14 — A helper to test predictions

`def predictor(input_ticket, org_label):` → takes a ticket **and** its true label, then compares.
- `print(f"Input Ticket: {input_ticket}")` → show the ticket
- `result = classifier(input_ticket)` → run the model → get a predicted label (as a number, like `LABEL_9`)
- `org_label_decoded = label_encoder.inverse_transform([int(org_label)])[0]` → turn the **true** number back into a department name
- `decoded_label = label_encoder.inverse_transform([int(result[0]['label'].split("_")[-1])])[0]`
  - `result[0]['label']` looks like `"LABEL_9"`; `.split("_")[-1]` grabs the `9`; then convert `9` → its department name
- The two `print(...)` lines show **Original vs Predicted** so you can see if the model got it right.

In [14]:
# Predictor Function to evaluate some tickets
def predictor(input_ticket,org_label):
    print(f"Input Ticket: {input_ticket}")
    result = classifier(input_ticket)
    print("\n")
    org_label_decoded = label_encoder.inverse_transform([int(org_label)])[0]
    decoded_label = label_encoder.inverse_transform([int(result[0]['label'].split("_")[-1])])[0]
    print("Original Label: ",org_label,",Original Label Decoded: ",org_label_decoded)
    print(f"Predicted Label: {int(result[0]['label'].split('_')[-1])} ,Predicted label Decoded: {decoded_label}")

### ✅ Cell 15 — Test ticket #319

Ticket: *"I am unable to connect to the Wi-Fi."*
- Original: **Customer Service** · Predicted: **Customer Service** → ✅ correct

In [15]:
predictor(df['text'][319],df['labels'][319])

Input Ticket: I am unable to connect to the Wi-Fi.


Original Label:  1 ,Original Label Decoded:  Customer Service
Predicted Label: 1 ,Predicted label Decoded: Customer Service


### ✅ Cell 16 — Test ticket #338

Ticket: IT consulting / server setup request.
- Original: **Technical Support** · Predicted: **Technical Support** → ✅ correct

In [16]:
predictor(df['text'][338],df['labels'][338])

Input Ticket: Dear Customer Support Team,

I am contacting you to seek prompt professional help regarding our IT Consulting Service. We are facing an urgent requirement for server setup and network enhancement. Our systems are presently experiencing difficulties that may negatively affect our business activities. It is imperative that we address these issues swiftly to avoid any interruptions.

Could you kindly prioritize our request and allocate an expert to help us with these concerns? We need someone with specialized expertise in server setups and optimization methods. Please inform us at your earliest convenience about the availability of your support personnel.

We are ready for a consultation call whenever it suits you to provide any additional information needed. You can reach me at <tel_num>.

Thank you for your prompt attention to this issue. We anticipate your swift reply.

Best regards,

<name>


Original Label:  9 ,Original Label Decoded:  Technical Support
Predicted Label:

### ✅ Cell 17 — Test ticket #317

Ticket: AWS billing discrepancy.
- Original: **Billing and Payments** · Predicted: **Billing and Payments** → ✅ correct

In [17]:
predictor(df['text'][317],df['labels'][317])

Input Ticket: Dear Customer Support,

I hope this message finds you well. I am writing to bring to your attention an issue concerning the recent billing related to our AWS cloud usage. Upon reviewing our most recent statement, it appears that there are discrepancies that have significantly impacted our cost estimates. It seems the charges associated with the AWS Management Service do not align with the actual usage recorded on our account <acc_num>.

The incorrect billing has resulted in unexpected costs that differ noticeably from our budget forecasts, making it difficult for us to manage our financial resources efficiently. This discrepancy was first noticed by <name> from our finance department, prompting an urgent need for your review of the billing details.

Could you please conduct a thorough review of our account to determine the cause of this miscalculation? We believe that an error in recording or processing has occurred that needs rectification. We would appreciate it if you 

### ✅ Cell 18 — Test ticket #210

Ticket: Cisco Router connectivity outage.
- Original: **Technical Support** · Predicted: **Technical Support** → ✅ correct

In [18]:
predictor(df['text'][210],df['labels'][210])

Input Ticket: Dear Customer Support,

I hope this message finds you well. I am writing to report a high-priority incident involving unstable connectivity issues with our Cisco Router ISR4331, which is currently impacting the performance of our enterprise network. Our entire network operations depend heavily on this router, and any disruptions can lead to significant operational setbacks.

The connectivity issues started occurring approximately 48 hours ago and have progressively worsened. Our IT team has conducted preliminary troubleshooting, which includes checking the physical connections, updating the firmware, and resetting the device multiple times; however, these actions have not resolved the issue. The router still exhibits sporadic connectivity drop-offs, causing disruptions in our daily workflows and negatively affecting the user experience within our enterprise.

We are requesting immediate technical assistance from your team to diagnose and resolve this matter as quickly as 

### ✅ Cell 19 — Test ticket #50

Ticket: Dell XPS performance issue after update.
- Original: **Technical Support** · Predicted: **Technical Support** → ✅ correct

In [19]:
predictor(df['text'][50],df['labels'][50])

Input Ticket: Hi, I've noticed performance issues with my Dell XPS 13 9310 after the latest update. Please assist.


Original Label:  9 ,Original Label Decoded:  Technical Support
Predicted Label: 9 ,Predicted label Decoded: Technical Support


### ✅ Cell 20 — Test ticket #175

Ticket: Epson printer frequent paper jams.
- Original: **Product Support** · Predicted: **Product Support** → ✅ correct

In [20]:
predictor(df['text'][175],df['labels'][175])

Input Ticket: Dear Customer Support Team,

I am writing to express my concerns regarding the Epson EcoTank ET-4760 printer that I purchased from your Tech Online Store. Despite being quite enthusiastic about its features, I have encountered frequent paper jams during printing operations, which significantly disrupt my workflow. This issue is hindering my productivity, and I would greatly appreciate your guidance on how to resolve it.

Could you please provide troubleshooting advice or recommend any steps I should take to remedy this situation? Additionally, if this is a known issue, kindly let me know if there is any update or technical support available to address it.

I am relying on your expertise to help find a suitable solution at your earliest convenience. Please feel free to contact me with any further instructions or if additional information is required for diagnostics on my end.

Thank you for your attention and assistance.

Best regards,

<name>


Original Label:  5 ,Origina

### ❌ Cell 21 — Test ticket #15

Ticket: HP printer won't connect to Wi-Fi.
- Original: **Customer Service** · Predicted: **Product Support** → ❌ **wrong**
- This is a good example of the model's limits: with only 288 training examples, some tickets get mis-routed (a printer issue *looks like* Product Support). More/cleaner data + tuning would improve this.

In [21]:
predictor(df['text'][15],df['labels'][15])

Input Ticket: Hello Customer Support,

I am experiencing a problem with my HP DeskJet 3755 printer. It fails to connect to the wireless network despite adhering to the setup guidelines. Could you provide troubleshooting support to resolve this issue?

Thank you, 
<name>


Original Label:  1 ,Original Label Decoded:  Customer Service
Predicted Label: 5 ,Predicted label Decoded: Product Support


---

# ❓ Q&A — Understanding the Libraries

## Q1: When scikit-learn "turns labels into numbers" — which labels, and what numbers?

**The labels = the department names** (the `labels` column — the thing we predict). The model only understands numbers, so `LabelEncoder` gives **each department a number**, in **alphabetical order**:

| Number | Department (label) |
|---|---|
| 0 | Billing and Payments |
| 1 | Customer Service |
| 2 | General Inquiry |
| 3 | Human Resources |
| 4 | IT Support |
| 5 | Product Support |
| 6 | Returns and Exchanges |
| 7 | Sales and Pre-Sales |
| 8 | Service Outages and Maintenance |
| 9 | Technical Support |

So `df['labels'] = label_encoder.fit_transform(df['labels'])` **replaces** each name with its number:
```
"Technical Support"    → 9
"Billing and Payments" → 0
"Product Support"      → 5
```
Later, `inverse_transform` reverses it (number → name) so you can read the prediction.

👉 **Only the labels get encoded — not the ticket text.** The text is handled separately by the *tokenizer*.

**One line:** LabelEncoder turns the department names into numbers 0–9 (alphabetical) so the model can process them — and reverses it for reading predictions.

## Q2: What is the difference between `torch` and `numpy`?

Both work with arrays of numbers and math — but for different purposes:

| | **numpy** | **torch** (PyTorch) |
|---|---|---|
| Main job | General number/array math | **Deep learning** (running/training AI models) |
| Runs on | **CPU only** | **CPU *and* GPU** (`cuda`) |
| Special power | Fast array math | **Autograd** — auto-computes gradients (needed to *train* models) |
| Its arrays are called | "arrays" (`ndarray`) | "tensors" |
| Role here | Small helper math | The engine that actually runs DeBERTa |

**Simple way to see it:**
- **numpy** = a great **calculator** for everyday number crunching
- **torch** = a calculator **built for AI** — uses the GPU *and* can learn (adjust weights during training)

numpy **can't train a neural network** (no GPU, no gradients) — that's why deep learning needs torch. (A torch "tensor" is basically a numpy array with superpowers.)

**One line:** numpy = general CPU calculator; torch = AI-focused calculator that uses the GPU and can *learn* (train models).

## Q3: Is pandas used to load the data?

**Yes! ✅** In this notebook:
```python
df = pd.read_csv("customer_tickets.csv")
```
`pandas` (`pd`) reads the CSV into a **table** (a "DataFrame") — rows & columns like a spreadsheet in code. It powers `df.columns = [...]`, `df.head()`, `df.dropna()`, etc.

There are **three data-handling libraries** here, each with a role:

| Library | Role in the notebook |
|---|---|
| **pandas** | **Loads** the CSV & cleans it (the spreadsheet-style table) |
| **datasets** (Hugging Face) | **Converts** that table into the format the trainer needs + splits train/test |
| **numpy** | Background number/array math (used indirectly) |

Flow: **pandas loads it → datasets prepares it for training.**

**One line:** Yes — pandas loads the CSV into a table; then datasets reshapes it for the model.

## Q4: What is `label_encoder.classes_`, and why `num_labels=len(label_encoder.classes_)`?

### What `label_encoder.classes_` is
After fitting, `.classes_` is the **list of unique labels** — your 10 departments, in order:
```python
['Billing and Payments', 'Customer Service', 'General Inquiry',
 'Human Resources', 'IT Support', 'Product Support',
 'Returns and Exchanges', 'Sales and Pre-Sales',
 'Service Outages and Maintenance', 'Technical Support']
```
So `len(label_encoder.classes_)` = **10**.

### Why the model needs `num_labels`
A classification model's final layer (the **classification head**) has **one output slot per category**. For 10 departments it needs **10 output slots**, each giving the probability the ticket belongs to that department.
```python
AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=10)
```
👉 Tells the model: *"Build a head with 10 outputs, because there are 10 departments."*

### Why `len(...)` instead of hardcoding `10`
- ✅ **Automatic** — counts the departments for you
- ✅ **Won't break** — if the dataset changes (department added/removed), the number updates automatically
- ✅ **No mismatch bugs** — model output count always matches the actual labels

### Important clarification: 1 head with 10 outputs (not 10 heads)
It's **one** classification head with **10 output slots** — like one scoreboard with 10 rows, not 10 separate scoreboards:
```
   ONE classification head
   ┌───────────────────────────┐
   │ Billing            →   2%  │
   │ Customer Service   →   5%  │
   │ ...                        │
   │ Technical Support  →  88%  │ ← highest wins
   └───────────────────────────┘
        (10 slots total)
```
For each ticket, that single head outputs 10 scores; the **highest score wins**.

**One line:** `label_encoder.classes_` is the list of the 10 departments, so `len(...)` = 10. Passing it as `num_labels` builds **one classification head with 10 output slots** (one score per department) — using `len(...)` keeps it automatic and prevents mismatches if the data changes.

## Q5: At model-loading, we only told it the *number* of departments — not *which* department. Correct?

**Exactly right — great catch!** 🎯 At this point the model **only knows the *number* (10), not *which* department each slot means.**

### What the model knows vs doesn't know (right now)

| Model knows ✅ | Model does NOT know ❌ |
|---|---|
| "I have **10** output slots" | That slot 0 = "Billing" |
| Slots are just named `LABEL_0` … `LABEL_9` | That slot 9 = "Technical Support" |

At load time the 10 slots are **generic and nameless** — literally `LABEL_0 … LABEL_9`.

### Where does the meaning come from? Two separate places

**1. The number ↔ name mapping lives in the `LabelEncoder`, NOT the model.**
```
0 → "Billing and Payments"
9 → "Technical Support"
```
That's why later you call `label_encoder.inverse_transform(...)` to turn `LABEL_9` back into "Technical Support." The model gives a **number**; the LabelEncoder supplies the **name**.

**2. Which slot means what is *learned during training*.**
During `trainer.train()`, the model sees `ticket text → 9`, `ticket text → 0`, etc. Over 10 epochs it learns *"tickets like this should score high on slot 9."* It never learns the *word* "Technical Support" — only the **pattern → slot-number** link.

### The full chain
```
Ticket text
   → MODEL → slot number (e.g. LABEL_9)      ← model only knows numbers
   → LabelEncoder.inverse_transform(9)
   → "Technical Support"                      ← name comes from here
```

### 💡 Pro tip (optional)
You can teach the model the names by passing `id2label` / `label2id` at load:
```python
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(label_encoder.classes_),
    id2label={i: name for i, name in enumerate(label_encoder.classes_)},
    label2id={name: i for i, name in enumerate(label_encoder.classes_)},
)
```
Then output shows `"Technical Support"` directly instead of `LABEL_9` — no `inverse_transform` needed. This notebook didn't do this, so it relies on the LabelEncoder to decode.

**One line:** Correct — at model-loading you only give it the *count* (10 slots named `LABEL_0`…`LABEL_9`); it doesn't know which department is which. The number→name mapping lives separately in the `LabelEncoder` (used later via `inverse_transform`), and *which slot means what* is learned during training. (Optionally `id2label`/`label2id` can bake names into the model.)

## Q6: What do the warnings during `trainer.train()` mean?

Both are **just warnings, not errors** — training runs fine.

### 1️⃣ The tokenizer / PAD/BOS/EOS message
```
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config...
Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}
```
**Meaning:** Earlier we set `tokenizer.pad_token = tokenizer.eos_token`, which made the tokenizer's special tokens differ slightly from the model config. Hugging Face just **synced them** — updating the model config to match.

**Special tokens** are behind-the-scenes markers: **PAD** = filler, **BOS** = beginning-of-sentence, **EOS** = end-of-sentence.

**Problem?** ❌ No — it's **informational** ("I noticed a mismatch and fixed it"). Nothing to do.

### 2️⃣ The `pin_memory` message ⚠️ (the important one)
```
'pin_memory' argument is set as true but no accelerator is found...
```
**Meaning:** **No GPU detected — you're training on CPU.** `pin_memory` is a speed trick that only helps with a GPU, so it's skipped.

**Problem?** Not an error — but CPU training is **slow** (especially 10 epochs of DeBERTa). It works, just takes much longer.

### 👉 Recommended: turn on the GPU
- **Kaggle:** Settings → Accelerator → **GPU** (e.g. T4), then re-run
- **Colab:** Runtime → Change runtime type → **GPU**, then re-run

Once a GPU is active, the `pin_memory` warning disappears and training is far faster.

### Summary

| Message | Type | Action |
|---|---|---|
| PAD/BOS/EOS tokens updated | ℹ️ Info | None — auto-fixed |
| `pin_memory ... no accelerator` | ⚠️ Warning | Optional but recommended: **enable GPU** |

**One line:** Both are harmless warnings — the first says HF auto-synced the tokenizer's special tokens (PAD/BOS/EOS) with the model config (because we set pad = eos earlier); the second says no GPU was found so you're on CPU (slower) — switch the accelerator to GPU to speed it up.

## Q7: Why save the tokenizer *alongside* the model?

### Core reason: the model learned to expect *this exact* tokenizer's numbers

Each tokenizer has its **own vocabulary** — its own word→number mapping. The model was **trained** using *that specific* numbering:
- During training it learned patterns like *"token number **2293** tends to mean X"*
- Those numbers came from **this** tokenizer

If you later load the model with a **different tokenizer**, the same word becomes a **different number**, and the model misreads everything. **Garbage in → garbage out.** 🗑️

### Analogy: a lock and its key 🔐
- **Model** = the lock (trained to open for specific number-patterns)
- **Tokenizer** = the key (produces those exact numbers)

Model + wrong tokenizer = wrong key = broken. So save them together as a **matched pair**.

### What would break without it
```
"login" → tokenizer A (training) → 5012
"login" → tokenizer B (wrong)    → 8834   ← different number!
```
The model learned patterns for `5012`, but now gets `8834` → nonsense predictions. The model is fine; the *inputs* are now in a different language.

### Bonus: you customized this tokenizer
In this notebook you changed it:
```python
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
```
Saving it preserves **your specific settings**, not just the defaults.

### That's why loading needs both
```python
classifier = pipeline("text-classification",
                      model="./text-classification-model",
                      tokenizer=tokenizer)
```
It needs **both** the model *and* a matching tokenizer. Saving them in the same folder keeps the pair together for a clean reload.

**One line:** You save the tokenizer with the model because the model was trained expecting *that exact* tokenizer's word→number mapping — a different tokenizer gives mismatched numbers and garbage output. They're a matched pair (lock + key) that must travel and reload together.

## Q8: We fine-tuned the model — did the tokenizer get fine-tuned too?

**The model was fine-tuned** ✅ — but the **tokenizer was NOT.** Easy to mix up, so here's the distinction:

| | Fine-tuned? | What happened |
|---|---|---|
| **The model** | ✅ **Yes** | `trainer.train()` adjusted its weights on your 288 tickets — it *learned* to route them |
| **The tokenizer** | ❌ **No** | Not trained by learning — only lightly *configured* (pad token, padding side) |

### The two save lines save different kinds of things
```python
trainer.save_model("./text-classification-model")        # saves the FINE-TUNED model (learned weights)
tokenizer.save_pretrained("./text-classification-model") # saves the tokenizer AS-IS (not fine-tuned)
```

### Why the tokenizer doesn't get fine-tuned
Fine-tuning = **adjusting the model's weights** to your task. The tokenizer has **no weights to learn** — it's just a fixed dictionary (word → number). It doesn't "improve" from training; it only *translates*.

> Analogy: you sent a student to a training course (the **model** learned). The **dictionary** they used didn't change — you just keep it with them so they can still read.

### So why save it at all?
Same reason as Q7 — the fine-tuned model still needs its **matching translator** to understand inputs. Even unchanged, you save it so model + tokenizer stay paired.

**One line:** Yes — `trainer.save_model` saves the *fine-tuned* model (weights actually trained on your data), but the tokenizer was *not* fine-tuned — it has no weights to learn; it's saved as-is (with your small config tweaks) so it stays paired with the model.

## Q9: But we didn't change the tokenizer's vocabulary — so what's the problem?

**Sharp reasoning — you're right.** In *your specific case*, since the vocabulary didn't change, there's **no real problem.** Let me reconcile this with Q7.

### You're correct: nothing broke here
Your tokenizer is **still the standard DeBERTa tokenizer** — you only tweaked `pad_token` and `padding_side`, not the vocabulary. So you **could** skip saving it and just reload the original:
```python
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-base")
```
…and it would produce the **exact same numbers** → your model works fine. **No garbage output in this case.**

### So what was Q7 warning about?
Q7's "garbage output" danger applies when you use a **genuinely different tokenizer** — e.g., a **BERT** tokenizer with your **DeBERTa** model. Those have different vocabularies → mismatched numbers → broken. Since you kept the same vocab, that danger **doesn't apply to you.** Q7 was the *general rule*; your case is the safe exception.

### Then why does the notebook still save it? (best practice)
Not required here — but good practice:

| Reason | Why it helps |
|---|---|
| **1. Self-contained folder** | Model + tokenizer in one place → easy to share/upload/reload |
| **2. Preserves your tweaks** | Saves your `pad_token` / `padding_side="right"` so you don't re-apply them |
| **3. Offline / version-safe** | Works without re-downloading; won't break if the original repo changes |

### Honest takeaway
- **Your case:** saving the tokenizer is a **convenience**, not a necessity (vocab unchanged).
- **In general:** always save it anyway — guarantees a matched, portable pair, and is foolproof for models where the tokenizer *was* customized.

**One line:** Correct — since the vocabulary is unchanged, there's no real problem; you could reload DeBERTa's original tokenizer and it'd match. Saving it alongside is best-practice convenience (self-contained, preserves your tweaks, offline-safe) — the Q7 "garbage output" risk only applies if you swap in a *different* tokenizer with a different vocabulary.

## Q10: Explain the `trainer.evaluate()` output

It's the model's "report card" on the 50 test tickets it never trained on.

### The warning (ignore it)
```
UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars...
```
**Harmless** ✅ — a technical note about collecting results across devices. Doesn't affect anything.

### The `[4/4 00:00]`
A **progress bar** — 4 evaluation batches (50 tickets ÷ batch size 8 ≈ 4) done in under a second.

### The metrics

| Metric | Value | Meaning |
|---|---|---|
| **`eval_loss`** | **1.39** | ⭐ The main one — **how wrong** the model is on test data. **Lower = better.** |
| `eval_runtime` | 1.14 | Took **1.14 seconds** to evaluate all 50 tickets |
| `eval_samples_per_second` | 43.67 | ~**44 tickets/second** (speed) |
| `eval_steps_per_second` | 3.49 | ~**3.5 batches/second** (speed) |
| `epoch` | 10.0 | Measured **after all 10 training rounds** |

### The one that matters: `eval_loss = 1.39`
**Loss = a "wrongness score."** The model outputs probabilities for the 10 departments; loss measures how far its guesses are from the correct answers. **0 = perfect; higher = more wrong.**

**Is 1.39 good?** ⚠️ Not great. A well-trained 10-class classifier usually scores well below 1.0 (often 0.3–0.7). At 1.39 the model learned *something* (most test predictions were right) but isn't confident/fully accurate — matching the wrong prediction you saw (printer → wrong department).

### Why the loss is high (honest reason)
- **Tiny dataset** — only 288 training tickets for 10 categories (~29 each)
- Some departments had only **2 examples** (General Inquiry, Human Resources) — nearly impossible to learn
- DeBERTa is big and usually needs **more data**

### Small note
`eval_loss` (1.39) is slightly **lower** than `train_loss` (1.60) — unusual (train is normally lower). With such a small dataset these numbers bounce around and aren't very reliable.

### How to improve it
- **More data** (biggest lever), **more epochs** / learning-rate tuning, a **smaller model** for small data, and **balance** the departments.

**One line:** `eval_loss = 1.39` is the model's "wrongness score" on the 50 unseen test tickets (lower = better) — fairly high, meaning it learned only partially, mainly due to the small dataset. The other numbers are speed stats, `epoch: 10.0` means measured after 10 rounds, and the warning is harmless.

## Q11: DeBERTa is an encoder and we use `AutoModelForSequenceClassification` — if I use a model that isn't a "classification model," will it error?

**Mostly right, with one nuance.** It's not that a model *is* or *isn't* a "classification model."

### Key idea: you *attach* a head to a base model
A model has a **body** (the Transformer) + a **head** (task output layer). `AutoModelForSequenceClassification` takes a model's body and **bolts on a classification head**. So DeBERTa isn't inherently a "classification model" — you're **turning it into one**.

### What happens with a different model? Three cases

| You use... | Result |
|---|---|
| **Another encoder** (BERT, RoBERTa, DistilBERT) | ✅ **Works great** — encoders are ideal for classification |
| **A decoder** (GPT-2, LLaMA) | ⚠️ **Often works, not ideal** — transformers supports classification heads on some decoders (e.g. `GPT2ForSequenceClassification`), so it may **not** error — but encoders are better suited |
| **A model with no classification support** | ❌ **Errors** — "Unrecognized configuration class ... for AutoModelForSequenceClassification" |

### Your statement, corrected
- ✅ DeBERTa (encoder) fits `AutoModelForSequenceClassification` well.
- ⚠️ A different model **doesn't automatically error** — it errors *only if* that architecture has **no SequenceClassification version** in the library. Many models (even some decoders) support it and would run.

### The real rule
It's about **whether the model architecture supports the head you're asking for** — not whether it's "a classification model." (Same pattern as forcing distilbert-sentiment into `text-generation`: it errored because it had no generation head.)

**One line:** Right that DeBERTa fits classification well — but a different model doesn't *automatically* error; `AutoModelForSequenceClassification` attaches a classification head to whatever model supports one (many encoders and some decoders do). It errors only if that architecture has no classification-head version in the library.

## Q12: Is `AutoModelForSequenceClassification` the head we're attaching?

**Almost — one precise tweak:** it isn't the head itself; it's the **loader** that gives you "base model + classification head" together.

### The precise breakdown
```
AutoModelForSequenceClassification
   = [ base model body (DeBERTa) ]  +  [ classification head ]
              ↑                              ↑
        understands text            the actual "head" (10 outputs)
```
- **The head** = the small final layer that outputs the 10 department scores
- **`AutoModelForSequenceClassification`** = the **loader/wrapper** that attaches that head onto the base model and hands you the whole thing

### In your code
```python
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=10)
```
= *"Load DeBERTa's body **and** attach a classification head with 10 outputs."* Result (`model`) = body + head together.

### Analogy 🔌
- **Base model (DeBERTa)** = a device
- **Classification head** = an attachment/adapter
- **`AutoModelForSequenceClassification`** = the tool that **plugs the attachment onto the device** for you

### The family (each attaches a different head)
| Class | Head it attaches |
|---|---|
| `AutoModelForSequenceClassification` | classification head (10 departments) ← yours |
| `AutoModelForTokenClassification` | NER head |
| `AutoModelForQuestionAnswering` | Q&A head |
| `AutoModelForCausalLM` | text-generation (LM) head |

**One line:** `AutoModelForSequenceClassification` is the *loader* that attaches a classification head to the base model and returns the combined thing. The "head" is the small 10-output layer; the class is what bolts it on — giving you DeBERTa body + classification head, assembled for you.

## Q13: Summary — `AutoModelForSequenceClassification` + how labels work in training

### Part A: What `AutoModelForSequenceClassification` is (summary)
- ✅ You put an **encoder-type Transformer** (like DeBERTa/BERT) into it
- ✅ It **attaches a classification head** on top
- ✅ The result can **classify things** — pick **one category** from a fixed list (e.g. route a ticket to 1 of 10 departments)

*(Encoders are the ideal choice; some decoders technically work too, but encoders are better for classification.)*

**Clean sentence:** `AutoModelForSequenceClassification` = take an encoder Transformer + attach a classification head → a model that sorts text into categories.

### Part B: During training, the model only knows numbers
- ✅ The model **never sees department names** during training — only **numbers** (0–9)
- ✅ It learns the pattern: *"this kind of ticket → slot number 9"*
- ✅ The **LabelEncoder** holds the number ↔ name mapping; `inverse_transform` turns the model's number **back into the department name** when we read the result

```
Training:   ticket → model learns → number (e.g. 9)          ← model only knows numbers
Reading it: number (9) → LabelEncoder → "Technical Support"  ← name comes from here
```

**Precision:** the model does the *thinking* (in numbers); the LabelEncoder does the *translating* (number → department name — it's just a lookup table).

**One line:** `AutoModelForSequenceClassification` puts an encoder + a classification head to sort text into categories; during training the model works purely in numbers, and the LabelEncoder is the lookup table that converts those numbers back into the actual department names.